### Data Ingestion For BSA 2023

In [1]:
import os
import re
import fitz
import numpy as np

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

from sentence_transformers import SentenceTransformer

from pymongo import MongoClient, ReplaceOne

c:\Users\Himanshu\Desktop\LexiBNS\.myenv\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(
c:\Users\Himanshu\Desktop\LexiBNS\.myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PDF_PATH = r"./THE_BHARATIYA_SAKSHYA_ADHINIYAM_2023.pdf"

LAW = "BSA"
MAX_SECTION = 170
VECTOR_INDEX_NAME = "test_bns"
DB_NAME = "test_db"
COLLECTION_NAME = "test_collection"

In [3]:
def extract_main_text(page):

    blocks = page.get_text("blocks")

    kept_blocks = []

    for block in blocks:

        x0, y0, x1, y1, text, *_ = block

        if (
            x0 >= 110
            and x1 <= 490
            and 80 <= y0 <= 780
        ):

            text = text.strip()

            if not text:
                continue

            upper = text.upper()

            if "THE GAZETTE OF INDIA" in upper:
                continue

            if "[PART II" in upper:
                continue

            kept_blocks.append(
                (y0, x0, text)
            )

    kept_blocks.sort(
        key=lambda x: (x[0], x[1])
    )

    return "\n".join(
        text
        for _, _, text in kept_blocks
    )

In [4]:
pdf = fitz.open(PDF_PATH)

full_text = ""

schedule_page = None

for page_number, page in enumerate(
    pdf,
    start=1
):

    raw_text = page.get_text("text")

    compact = re.sub(
        r"\s+",
        "",
        raw_text
    ).upper()

    # Actual Schedule starts on its own page.
    # Avoid false match from references to
    # "the Schedule" inside Section 63.
    if compact.startswith(
        "THESCHEDULE[SEESECTION63(4)(C)]"
    ):

        schedule_page = page_number
        break

    page_text = extract_main_text(
        page
    )

    full_text += (
        f"\n<<<PAGE:{page_number}>>>\n"
        + page_text
        + "\n"
    )


print("Total PDF pages:", len(pdf))
print("Schedule begins on page:", schedule_page)
print("Statute characters:", len(full_text))

Total PDF pages: 47
Schedule begins on page: 46
Statute characters: 139658


In [5]:
def parse_bsa_sections(
    text,
    pdf_path
):

    starts = []

    search_position = 0

    for section_number in range(
        1,
        MAX_SECTION + 1
    ):

        pattern = re.compile(
            rf"(?m)^\s*{section_number}\.\s*"
        )

        match = pattern.search(
            text,
            search_position
        )

        if match is None:

            raise ValueError(
                f"BSA Section "
                f"{section_number} not found"
            )

        starts.append(
            (
                section_number,
                match
            )
        )

        search_position = (
            match.end()
        )

    sections = []

    for i, (
        section_number,
        match
    ) in enumerate(starts):

        start = match.start()

        if i + 1 < len(starts):

            end = starts[
                i + 1
            ][1].start()

        else:

            end = len(text)

        raw_section = text[
            start:end
        ]

        pages = [
            int(page)
            for page in re.findall(
                r"<<<PAGE:(\d+)>>>",
                raw_section
            )
        ]

        clean_text = re.sub(
            r"<<<PAGE:\d+>>>",
            "",
            raw_section
        ).strip()

        sections.append(
            Document(
                page_content=
                    clean_text,

                metadata={
                    "law":
                        "BSA",

                    "section":
                        str(section_number),

                    "pages":
                        sorted(
                            set(pages)
                        ),

                    "source":
                        pdf_path,

                    "document_type":
                        "primary_statute"
                }
            )
        )

    return sections

In [6]:
sections = parse_bsa_sections(
    full_text,
    PDF_PATH
)

print(
    "Sections extracted:",
    len(sections)
)

Sections extracted: 170


In [7]:
actual_sections = [

    int(
        doc.metadata["section"]
    )

    for doc in sections
]

expected_sections = list(
    range(1, 171)
)


print(
    "All sections valid:",
    actual_sections
    == expected_sections
)

print(
    "First 10:",
    actual_sections[:10]
)

print(
    "Last 10:",
    actual_sections[-10:]
)


assert (
    actual_sections
    == expected_sections
)

All sections valid: True
First 10: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Last 10: [161, 162, 163, 164, 165, 166, 167, 168, 169, 170]


In [8]:
def show_section(number):

    for doc in sections:

        if (
            doc.metadata["section"]
            == str(number)
        ):

            print(
                "LAW:",
                doc.metadata["law"]
            )

            print(
                "SECTION:",
                doc.metadata["section"]
            )

            print(
                "PAGES:",
                doc.metadata["pages"]
            )

            print("\nTEXT:\n")

            print(
                doc.page_content
            )

            return

In [9]:
show_section(23)

LAW: BSA
SECTION: 23
PAGES: [12]

TEXT:

23. (1) No confession made to a police officer shall be proved as against a person
accused of any offence.
(2) No confession made by any person while he is in the custody of a police officer,
unless it is made in the immediate presence of a Magistrate shall be proved against him:


Provided that when any fact is deposed to as discovered in consequence of information
received from a person accused of any offence, in the custody of a police officer, so much
of such information, whether it amounts to a confession or not, as relates distinctly to the
fact discovered, may be proved.


In [10]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print(
    "Embedding dimension:",
    model.get_sentence_embedding_dimension()
)

print(
    "Model max sequence:",
    model.max_seq_length
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1561.83it/s]


Embedding dimension: 384
Model max sequence: 256


C:\Users\Himanshu\AppData\Local\Temp\ipykernel_20916\494620326.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension()


In [11]:
splitter = RecursiveCharacterTextSplitter(

    chunk_size=650,

    chunk_overlap=80,

    separators=[
        "\n\n",
        "\nExplanation",
        "\nIllustration",
        "\nIllustrations",
        "\nProvided",
        "\n(",
        ". ",
        " "
    ]
)

In [12]:
def token_count(text):

    return len(
        model.tokenizer.tokenize(
            text
        )
    )

In [13]:
def create_chunks(
    sections
):

    final_chunks = []

    for section in sections:

        section_text = (
            section.page_content
        )

        # Small provision
        if (
            token_count(section_text)
            <= 180
        ):

            parts = [
                section_text
            ]

        # Long provision
        else:

            parts = (
                splitter.split_text(
                    section_text
                )
            )

        safe_parts = []

        for part in parts:

            if (
                token_count(part)
                <= 210
            ):

                safe_parts.append(
                    part
                )

            else:

                smaller_splitter = (
                    RecursiveCharacterTextSplitter(

                        chunk_size=400,

                        chunk_overlap=50,

                        separators=[
                            "\n\n",
                            ". ",
                            " "
                        ]
                    )
                )

                safe_parts.extend(
                    smaller_splitter
                    .split_text(part)
                )

        for index, part in enumerate(
            safe_parts
        ):

            final_chunks.append(
                Document(
                    page_content=part,

                    metadata={
                        **section.metadata,

                        "parent_section":
                            section.metadata[
                                "section"
                            ],

                        "chunk_index":
                            index
                    }
                )
            )

    return final_chunks

In [14]:
chunks = create_chunks(
    sections
)

print(
    "Parent sections:",
    len(sections)
)

print(
    "Child chunks:",
    len(chunks)
)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1083 > 256). Running this sequence through the model will result in indexing errors


Parent sections: 170
Child chunks: 327


In [15]:
def build_embedding_text(
    chunk
):

    return f"""
Bharatiya Sakshya Adhiniyam, 2023
BSA Section {chunk.metadata['section']}

{chunk.page_content}
""".strip()

In [16]:
invalid_chunks = []

max_tokens = 0

largest_chunk = None


for chunk in chunks:

    text = build_embedding_text(
        chunk
    )

    count = token_count(
        text
    )

    if count > max_tokens:

        max_tokens = count

        largest_chunk = chunk

    if (
        count
        > model.max_seq_length
    ):

        invalid_chunks.append({

            "section":
                chunk.metadata[
                    "section"
                ],

            "chunk":
                chunk.metadata[
                    "chunk_index"
                ],

            "tokens":
                count
        })


print(
    "Maximum embedding tokens:",
    max_tokens
)

print(
    "Model limit:",
    model.max_seq_length
)

print(
    "Invalid chunks:",
    len(invalid_chunks)
)

print(
    invalid_chunks[:10]
)


assert (
    len(invalid_chunks)
    == 0
)

Maximum embedding tokens: 194
Model limit: 256
Invalid chunks: 0
[]


In [17]:
embedding_texts = [

    build_embedding_text(
        chunk
    )

    for chunk in chunks
]

In [18]:
embeddings = model.encode(

    embedding_texts,

    batch_size=32,

    normalize_embeddings=True,

    show_progress_bar=True
)

Batches: 100%|██████████| 11/11 [00:09<00:00,  1.10it/s]


In [20]:
print(
    "Chunks:",
    len(chunks)
)

print(
    "Embeddings:",
    len(embeddings)
)

print(
    "Shape:",
    embeddings.shape
)


assert (
    len(chunks)
    == len(embeddings)
)

Chunks: 327
Embeddings: 327
Shape: (327, 384)


In [21]:
MONGO_URI = os.getenv(
    "MONGODB_URI"
)

client = MongoClient(
    MONGO_URI
)

db = client[
    DB_NAME
]

collection = db[
    COLLECTION_NAME
]


print(
    client.admin.command(
        "ping"
    )
)

{'ok': 1}


In [22]:
indexes = list(
    collection.list_search_indexes()
)

for index in indexes:

    print(
        "Name:",
        index.get("name")
    )

    print(
        "Status:",
        index.get("status")
    )

    print(
        "Queryable:",
        index.get("queryable")
    )

    print("-" * 50)

Name: test_bns
Status: READY
Queryable: True
--------------------------------------------------


In [23]:
mongo_documents = []


for i, chunk in enumerate(
    chunks
):

    section = (
        chunk.metadata[
            "section"
        ]
    )

    chunk_index = (
        chunk.metadata[
            "chunk_index"
        ]
    )


    mongo_documents.append({

        "id":
            f"bsa-{section}-{chunk_index}",

        "values":
            embeddings[i].tolist(),

        "metadata": {

            "law":
                "BSA",

            "section":
                section,

            "parent_section":
                section,

            "chunk_index":
                chunk_index,

            "document_type":
                "primary_statute",

            "text":
                chunk.page_content,

            "pages":
                chunk.metadata[
                    "pages"
                ],

            "source":
                os.path.basename(
                    PDF_PATH
                )
        }
    })


print(
    "Prepared:",
    len(mongo_documents)
)

Prepared: 327


In [24]:
collection.create_index(
    "id",
    unique=True
)

'id_1'

In [25]:
operations = [

    ReplaceOne(
        {
            "id":
                doc["id"]
        },

        doc,

        upsert=True
    )

    for doc in mongo_documents
]

In [26]:
result = collection.bulk_write(
    operations
)


print(
    "BSA records in MongoDB:",
    collection.count_documents(
        {
            "metadata.law":
                "BSA"
        }
    )
)

BSA records in MongoDB: 327


In [27]:
section_23 = list(
    collection.find(
        {
            "metadata.law":
                "BSA",

            "metadata.section":
                "23"
        },

        {
            "_id": 0
        }
    )
)


print(
    "Section 23 chunks:",
    len(section_23)
)


for doc in section_23:

    print(
        "\n",
        doc["metadata"]["text"]
    )

Section 23 chunks: 1

 23. (1) No confession made to a police officer shall be proved as against a person
accused of any offence.
(2) No confession made by any person while he is in the custody of a police officer,
unless it is made in the immediate presence of a Magistrate shall be proved against him:


Provided that when any fact is deposed to as discovered in consequence of information
received from a person accused of any offence, in the custody of a police officer, so much
of such information, whether it amounts to a confession or not, as relates distinctly to the
fact discovered, may be proved.


In [28]:
def mongo_search(
    query,
    k=5
):

    query_vector = model.encode(

        [query],

        normalize_embeddings=True

    )[0].tolist()


    pipeline = [

        {
            "$vectorSearch": {

                "index":
                    "test_bns",

                "path":
                    "values",

                "queryVector":
                    query_vector,

                "numCandidates":
                    max(
                        100,
                        k * 20
                    ),

                "limit":
                    k,

                "filter": {
                    "metadata.law":
                        "BSA"
                }
            }
        },

        {
            "$project": {

                "_id": 0,

                "id": 1,

                "metadata": 1,

                "score": {
                    "$meta":
                        "vectorSearchScore"
                }
            }
        }
    ]


    return list(
        collection.aggregate(
            pipeline
        )
    )

In [29]:
results = mongo_search(

    "Can a confession made to "
    "a police officer be used "
    "against an accused?",

    k=5
)


print(
    "Retrieved:",
    len(results)
)

Retrieved: 5


In [30]:
for result in results:

    print(
        "\nLAW:",
        result[
            "metadata"
        ]["law"]
    )

    print(
        "SECTION:",
        result[
            "metadata"
        ]["section"]
    )

    print(
        "SCORE:",
        round(
            result["score"],
            4
        )
    )

    print(
        result[
            "metadata"
        ]["text"][:500]
    )

    print(
        "=" * 80
    )


LAW: BSA
SECTION: 23
SCORE: 0.8502
23. (1) No confession made to a police officer shall be proved as against a person
accused of any offence.
(2) No confession made by any person while he is in the custody of a police officer,
unless it is made in the immediate presence of a Magistrate shall be proved against him:


Provided that when any fact is deposed to as discovered in consequence of information
received from a person accused of any offence, in the custody of a police officer, so much
of such information, whether it amounts 

LAW: BSA
SECTION: 22
SCORE: 0.8283
22. A confession made by an accused person is irrelevant in a criminal proceeding, if
the making of the confession appears to the Court to have been caused by any inducement,
threat, coercion or promise having reference to the charge against the accused person,
proceeding from a person in authority and sufficient, in the opinion of the Court, to give the
accused person grounds which would appear to him reasonable for suppos

In [31]:
def safe_mongo_search(
    query,
    k=5
):

    results = mongo_search(
        query,
        k
    )

    if len(results) == 0:

        raise RuntimeError(
            "MongoDB returned 0 BSA chunks. "
            "Do not call the LLM. "
            "Check collection and test_bns index."
        )

    return results

In [32]:
def extract_section_number(
    query
):

    match = re.search(

        r"\b(?:BSA\s+)?section\s+"
        r"(\d{1,3})\b",

        query,

        re.IGNORECASE
    )

    if match:

        return (
            match.group(1)
        )

    return None

In [33]:
def retrieve_bsa(
    query,
    k=5
):

    section_number = (
        extract_section_number(
            query
        )
    )


    # Exact lookup
    if section_number:

        results = list(
            collection.find(

                {
                    "metadata.law":
                        "BSA",

                    "metadata.section":
                        section_number
                },

                {
                    "_id": 0
                }
            )
        )


        if not results:

            raise RuntimeError(
                f"BSA Section "
                f"{section_number} "
                "not found."
            )


        return results


    # Semantic retrieval
    return safe_mongo_search(
        query,
        k
    )

In [34]:
from langchain_groq import ChatGroq


llm = ChatGroq(

    groq_api_key=
        os.getenv(
            "GROQ_API_KEY"
        ),

    model_name=
        "openai/gpt-oss-120b",

    temperature=0
)

In [35]:
def answer_bsa(
    question,
    k=5
):

    results = retrieve_bsa(
        question,
        k=k
    )


    context_parts = []


    for result in results:

        metadata = (
            result["metadata"]
        )


        context_parts.append(

            f"""
Bharatiya Sakshya Adhiniyam, 2023
BSA Section {metadata['section']}

{metadata['text']}
""".strip()
        )


    context = (
        "\n\n---\n\n".join(
            context_parts
        )
    )


    if not context.strip():

        raise RuntimeError(
            "Context is empty. "
            "LLM call cancelled."
        )


    print(
        "RETRIEVED SECTIONS:"
    )


    for result in results:

        print(
            "BSA Section",
            result[
                "metadata"
            ]["section"]
        )


    prompt = f"""
You are a legal information assistant.

Answer using ONLY the supplied provisions
from the Bharatiya Sakshya Adhiniyam, 2023.

Rules:

1. Mention the relevant BSA section number.

2. Do not invent evidentiary rules,
exceptions or legal provisions.

3. Use only the supplied context.

4. If none of the supplied provisions
contains the answer, say:
"Insufficient information in retrieved context."


CONTEXT:

{context}


QUESTION:

{question}


ANSWER:
"""


    response = llm.invoke(
        prompt
    )


    print(
        "\nANSWER:\n"
    )

    print(
        response.content
    )


    return response.content

In [36]:
answer_bsa(
    "Can a confession made to a police officer "
    "be used as evidence against the accused?"
)

RETRIEVED SECTIONS:
BSA Section 23
BSA Section 22
BSA Section 22
BSA Section 22
BSA Section 147

ANSWER:

**BSA Section 23(1)** states:

> “No confession made to a police officer shall be proved as against a person accused of any offence.”

Accordingly, a confession that is given to a police officer cannot be used as evidence against the accused. (The provision does allow a confession made while in police custody to be proved only if it is made in the immediate presence of a Magistrate, but a mere confession to a police officer is inadmissible.)


'**BSA Section\u202f23(1)** states:\n\n> “No confession made to a police officer shall be proved as against a person accused of any offence.”\n\nAccordingly, a confession that is given to a police officer cannot be used as evidence against the accused. (The provision does allow a confession made while in police custody to be proved only if it is made in the immediate presence of a Magistrate, but a mere confession to a police officer is inadmissible.)'

In [37]:
answer_bsa(
    "Explain BSA Section 23."
)

RETRIEVED SECTIONS:
BSA Section 23

ANSWER:

**BSA Section 23**

- **Section 23(1)** – A confession that is made **to a police officer** cannot be used as evidence against the accused person.

- **Section 23(2)** – A confession made **while the accused is in police custody** is also inadmissible, **unless the confession is made in the immediate presence of a Magistrate**.

- **Exception** – If, during police custody, the accused provides information that leads the police to discover a fact, the portion of that information that **directly relates to the discovered fact** (whether it is a confession or not) may be proved as evidence of that fact.

In short, Section 23 bars the use of confessions obtained by police officers, except when the confession is given before a Magistrate, and it allows only the specific factual disclosures that result in the discovery of a fact to be admitted.


'**BSA Section 23**\n\n- **Section\u202f23(1)** – A confession that is made **to a police officer** cannot be used as evidence against the accused person.\n\n- **Section\u202f23(2)** – A confession made **while the accused is in police custody** is also inadmissible, **unless the confession is made in the immediate presence of a Magistrate**.\n\n- **Exception** – If, during police custody, the accused provides information that leads the police to discover a fact, the portion of that information that **directly relates to the discovered fact** (whether it is a confession or not) may be proved as evidence of that fact.\n\nIn short, Section\u202f23 bars the use of confessions obtained by police officers, except when the confession is given before a Magistrate, and it allows only the specific factual disclosures that result in the discovery of a fact to be admitted.'